# Documents Notebook

For this chatbot I will be using bash man pages, tealdeer output and the output from the bash help system for builtin commands.

There will need to be 2 pipelines.
- The first will be the data ingestion pipeline, which is in this notebook.
- Then we will need a query pipeline that will use our domain specific emnbeddings as the context for an LLM prompt. This is the RAG implementation that will be built. It is located in bashbot.ipynb and the associated script.

The dependencies are in requirements.txt.

## Data Ingestion Pipeline for Bashbot

### Imports

In [1]:
import os
import sys
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader #,PyPDFLoader, PyMuPDFLoader for PDFs in the future
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Function Definitions

### load the corpus from the text files in the data directory

In [2]:
def load_txt_corpus():
    """ Loads all text files from the ./data directory."""
    loader = DirectoryLoader(
        "../data",
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'},
        show_progress=True
    )
    corpus = loader.load()

    return corpus

In [19]:
def process_text_files(path):
    """Processes all text (.txt) files in the provided path"""
    documents = []
    file_dir = Path(path)

    files = list(file_dir.glob("**/*.txt"))
    print(f"Preparing to process {len(files)} text files...")

    # process the files and modify the source metadata to include manpage reference if possible.
    for f in files:
#        print(f"processing {f.name}")
        try:
            loader = TextLoader(str(f))
            docs = loader.load()

            for d in docs:
                d.metadata['manpage'] = Path(f).stem
                d.metadata['source_file'] = f.name
                d.metadata['file_type'] = 'txt'
                d.metadata['platform'] = 'linux'
                d.metadata['source'] = 'manpage'

                documents.extend(docs)

        except Exception as e:
            print(f"    ERROR {e}")

    print(f"Total documents loaded: {len(documents)}.")
    print(type(documents))
    return list(documents)
    

In [20]:
def process_all_files():
    documents = []
    documents.extend(process_text_files("../data"))

    return documents

### divide text into chunks

In [26]:
# Splits the douments into manageable sized chunks.
def split_documents(corpus, chunk_size=500, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(corpus)
    print(f"Splitting {len(corpus)} documents into {len(split_docs)} chunks.")
    if split_docs:
        print("\nSample chunk:")
        print(f"Page_content: {split_docs[0].page_content[:300]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

### main()

In [22]:
# Obligatory entry point
def main():
   # load_txt_corpus()
    corpus = process_all_files()

In [23]:
#if __name__ == "__main__":
#   main()

## Notebook Exploration

**Read data from files, and maybe an RDB**

In [24]:
corpus = process_all_files()

Preparing to process 2078 text files...
Total documents loaded: 2078.
<class 'list'>


**Chunking**

In [27]:
chunks = split_documents(corpus)

Splitting 2078 documents into 39775 chunks.

Sample chunk:
Page_content: accessdb - dumps the content of a man-db database in a human readable format
will output the data contained within a man-db database in a human readable
form.  By default, it will dump the data from where <db-type> is dependent on
the database library in use....
Metadata: {'source': 'manpage', 'manpage': 'accessdb', 'source_file': 'accessdb.txt', 'file_type': 'txt', 'platform': 'linux'}


In [28]:
chunks[3]

Document(metadata={'source': 'manpage', 'manpage': 'aclocal-1.18', 'source_file': 'aclocal-1.18.txt', 'file_type': 'txt', 'platform': 'linux'}, page_content='--system-acdir=DIR directory holding third-party system-wide files\n--diff[=COMMAND] run COMMAND [diff -u] on M4 files that would be changed\n(implies --install and --dry-run) --dry-run pretend to, but do not actually\nupdate any file --force always update output file --help print this help, then\nexit -I DIR add directory to search list for .m4 files --install copy\nthird-party files to the first -I directory --output=FILE put output in FILE')

### Embedding and Vector DB store

## Prompt and RAG Retrieval Pipeline